In [ ]:
#New Testing script that uses new testing environment to run reproducable and fair experiments
from Single_Agent_Testing_Environment import SingleSatelliteEnvTest#Import the environment
from gymnasium.utils.env_checker import check_env#Import check environment functionality
import traceback
import numpy as np
import gymnasium as gym
import torch
print(type(SingleSatelliteEnvTest))
# This will catch many common issues
#Use dummy inputs
env = SingleSatelliteEnvTest(render_mode="None")
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print("Environment has issues:")
    traceback.print_exc()
from cleanrl.cleanrl.dqn import QNetwork#Import Q network from DQN script
from cleanrl.cleanrl.c51 import QNetwork as C51Network#Import Q network from C51 script
from cleanrl.cleanrl.ppo import Agent#Import Agent from PPO script
env =SingleSatelliteEnv(render_mode="None")
envs = gym.vector.SyncVectorEnv([
    lambda: SingleSatelliteEnvTest(render_mode="None")#Setup vectorized environment (might not be necessary)
])


In [ ]:

CGR_reward_vector=np.zeros([len(seeds)*5,])
CGR_delivery_ratio_vector=np.zeros([len(seeds)*5,])
CGR_mean_contact_energy_efficiency_vector=np.zeros([len(seeds)*5,])
CGR_initial_data_volume=np.zeros([len(seeds)*5,])
Sort_reward_vector=np.zeros([len(seeds)*5,])
Sort_delivery_ratio_vector=np.zeros([len(seeds)*5,])
Sort_mean_contact_energy_efficiency_vector=np.zeros([len(seeds)*5,])
Sort_initial_data_volume=np.zeros([len(seeds)*5,])

#Here we setup the testing loop
seeds=[0,1,2,3,4,5,6,7,8,9,10]
for seed in seeds:
    #We will use the seed to determine the initial data volume, weather conditions and link availability across all the algorithms
    np.np_random=np.random.default_rng()
    np.random(seed)
    #First we determine the initial data volume
    initial_data_volume=np.array(self.np_random.integers(5, 100) / 100.0, dtype=np.float32)
    #Second we determine the weather conditions for each 
    weather_conditions=np.random.uniform(low=0.0, high=1.0, size=(10,)).astype(np.float32)#Randomly initialize the weather conditions
    weather_conditions= np.round(weather_conditions*10)/10
    #Third we calculate link availability at each of these contacts
    link_availability=np.zeros([10,1])
    
    for i in range(0,9):
        link_availability[i], unavailable_time[i]=LinkAvail(weather_conditions[i],length)
    #Next we utilize our modified environment to test each of the schemes across all the test seeds
    C51_tr,C51_dr,C51_ec=
    
        
def LinkAvail(self,weather,length):
        delivered_packets=0#Local variable, number of packets delivered during the contact
        excess_energy_expended=0#Local variable, excess energy expended during the contact
        random_sample=self.np_random.uniform(low=0.0, high=1.0, size=(length,)).astype(np.float32)#10 random samples that we will compare to the cloud cover value
        remaining_data=np.round(remaining_data)#just rounding to be safe (probably should be handling the types better)
        for i in range(0,length):
            if random_sample[i]> weather:
                delivered_packets=delivered_packets+1#successfully transmit packet
                remaining_data=remaining_data-1
            else:
                excess_energy_expended=excess_energy_expended+1#not successful in transmission
           
        return delivered_packets,excess_energy_expended
    

In [ ]:
#Next we evaluate the results
#Produce box and whisker plots for each of the values




In [1]:
#Put the functions here
def C51_test(testing_seed, num_of_models,weather,link_avail,initial_data):
    envs = gym.vector.SyncVectorEnv([
        lambda: SingleSatelliteEnv(render_mode="None")
    ])#vectorize environment
    #Put the output vectors here for the tests
    vector_index=0#index for the different metric vectors
    C51_reward_vector=np.zeros([num_of_models,])#Reward vector initialization
    C51_delivery_ratio_vector=np.zeros([num_of_models,])#Delivery Ratio vector initialization
    C51_mean_contact_energy_efficiency_vector=np.zeros([num_of_models,])#Mean Contact Energy Efficiency vector initialization
    
            
    for k in range(0,num_of_models):
        init_obs,init_info=envs.reset(seed=testing_seed,options=[weather,link_avail,initial_data])
        
        model = C51Network(envs).to("cpu")
        
        # Load trained weights
        checkpoint = torch.load("Models/C51/c51_"+str(k)+".cleanrl_model")#We retrieve the checkpoint file
        model.load_state_dict(checkpoint["model_weights"]) #We get the model weights
        obs=init_obs.copy()#We use copy to be safe
        done = False
        total_reward=0
        while not done:
            obs_tensor = torch.tensor(obs, dtype=torch.float32)#We get the tensor from the observation space
            with torch.no_grad():
                action, q_values = model.get_action(obs_tensor)#We input the tensor in the get_action function to get the action from the model
                action=int(action)#Convert the action to an int
            obs, reward, terminated, truncated, info = envs.step([action])#Get next obs, reward...
            total_reward+=reward#We add to the total reward
            done = terminated or truncated#Check if the episode is over
        
        final_info=info['final_info'][0]#Get the final info 
        #print(final_info)
        initial_data_volume=final_info['Initial Data']#Get initial data that we needed to send
        #print("c51 initial data")
        #print(initial_data_volume)
        delivered_packets=final_info['Delivered Data']#Get the number of packets delivered during the episode
        #print("c51 delivered_packets")
        #print(delivered_packets)
        number_of_contacts=final_info['Number of Contacts']#Get the number of contacts that were usedi n hte episode
        excess_energy_expended=final_info['Energy Expended']#Get the energy expended 
        C51_reward_vector[vector_index]=total_reward#Update the reward vector with the total reward for this model and seed
        C51_delivery_ratio_vector[vector_index]=delivered_packets/(np.round(initial_data_volume*100))#We calculate delivery ratio by dividing the number of delivered packets by the packets 
        if number_of_contacts>0:
            C51_mean_contact_energy_efficiency_vector[vector_index]=delivered_packets/(number_of_contacts*10)#Calculate the mean energy efficiency
        else:
            C51_mean_contact_energy_efficiency_vector[vector_index]=0#We set the energy efficiency to zero if we don't use any contacts
        C51_initial_data_volume[vector_index]=np.round(initial_data_volume*100)#add to the initial data volume vector
        if(np.round(initial_data_volume*100)<delivered_packets):
            print("Delivery ratio error C51")
            print(initial_data_volume*100)
            print(delivered_packets)
        vector_index+=1#Increment index
    envs.close()
    return C51_reward_vector,C51_delivery_ratio_vector,C51_mean_contact_energy_efficiency_vector,C51_initial_data_volume#Output metrics
def DQN_test(testing_seed,num_of_test_episodes,num_of_models):
    envs = gym.vector.SyncVectorEnv([
        lambda: SingleSatelliteEnv(render_mode="None")
    ])#Convert to vectorized environment (not sure if this is necessary
    DQN_reward_vector=np.zeros([num_of_models,])#Setup the metric outputs
    DQN_delivery_ratio_vector=np.zeros([num_of_models,])#Setup metric output
    DQN_mean_contact_energy_efficiency_vector=np.zeros([num_of_models,])#Setup metric output
    DQN_initial_data_volume=np.zeros([num_of_models,])#Setup metric output
    vector_index=0#Initialize vector index
    for k in range(0,num_of_models):#Test all the models
        obs_init,init_info=envs.reset(seed=Testing_seed_Configuration[i])#Set the seed for the experiment
        model = QNetwork(envs).to("cpu")#Get the Q Network from the model
        checkpoint = torch.load("Models/DQN/dqn_"+str(k)+".cleanrl_model")#Load the specific checkpoint
        model.load_state_dict(checkpoint)#Get the model
        obs=obs_init.copy()#Copy to be safe
        done = False
        total_reward=0
        while not done:
            obs_tensor = torch.tensor(obs, dtype=torch.float32)#Get tensor 
            with torch.no_grad():
                q_values = model(torch.Tensor(obs).to("cpu"))#get the qvalues by inputting the tensor into the model
                action= torch.argmax(q_values, dim=1).cpu().numpy()#get the action by taking getting the argmax of the qvalues
            obs, reward, terminated, truncated, info = envs.step(action)#Take action and get next obs, reward and termination conditions
            total_reward+=reward#update total reward
            done = terminated or truncated#Check if the episode is over
        final_info=info['final_info'][0]#Get data from final step
        initial_data_volume=final_info["Initial Data"]#Get the initial data
        delivered_packets=final_info["Delivered Data"]#Get the number of delivered packets
        number_of_contacts=final_info["Number of Contacts"]#Get the number of contacts used
        excess_energy_expended=final_info["Energy Expended"]#Get the energy expended over the entire episode
        DQN_reward_vector[vector_index]=total_reward#note total reward
        DQN_delivery_ratio_vector[vector_index]=delivered_packets/(np.round(initial_data_volume*100))#Delivery Ratio is the number of delivered packets divided by the intial data volume
        if number_of_contacts>0:
            DQN_mean_contact_energy_efficiency_vector[vector_index]=delivered_packets/(number_of_contacts*10)#Calculate energy efficiency
        else:
            DQN_mean_contact_energy_efficiency_vector[vector_index]=0
        DQN_initial_data_volume[vector_index]=initial_data_volume*100
        if(np.round(initial_data_volume*100)<delivered_packets):
            print("Delivery ratio error DQN")
            print(initial_data_volume*100)
            print(delivered_packets)
        vector_index+=1
    envs.close()
    return DQN_reward_vector,DQN_delivery_ratio_vector,DQN_mean_contact_energy_efficiency_vector,DQN_initial_data_volume
def PPO_test(Testing_seed_Configuration,num_of_test_episodes,num_of_models):
    envs = gym.vector.SyncVectorEnv([
        lambda: SingleSatelliteEnv(render_mode="None")
    ])#Convert to vectorized environment
    PPO_reward_vector=np.zeros([num_of_models,])#Initializing main variables to collect metrics
    PPO_delivery_ratio_vector=np.zeros([num_of_models,])
    PPO_mean_contact_energy_efficiency_vector=np.zeros([num_of_models,])
    PPO_initial_data_volume=np.zeros([num_of_models,])
    vector_index=0
    for k in range(0,num_of_models):
        obs_init,init_info=envs.reset(seed=testing_seed,options=[])#Reset environment according to predefined seed
        agent = Agent(envs).to("cpu")#Get the agent (policy)
        checkpoint = torch.load("Models/PPO/ppo_"+str(k)+".cleanrl_model")#Import the checkpoint that we saved
        agent.load_state_dict(checkpoint["model"])#Get the agent from the checkpoint
        obs = obs_init.copy()#Copy obs to be safe
        done = False
        total_reward=0
        while not done:
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)#Make obs a tensor)
            with torch.no_grad():
                action, _, _,qvalue = agent.get_action_and_value(obs_tensor)#Get the action using the action_value function
            obs, reward, terminated, truncated, info = envs.step(action)
            total_reward+=reward
            done = terminated or truncated
        final_info=info['final_info'][0]#Get metrics from final step of episode
        initial_data_volume=final_info["Initial Data"]
        delivered_packets=final_info["Delivered Data"]
        number_of_contacts=final_info["Number of Contacts"]
        excess_energy_expended=final_info["Energy Expended"]
        PPO_reward_vector[vector_index]=total_reward
        PPO_delivery_ratio_vector[vector_index]=delivered_packets/(np.round(initial_data_volume*100))
        if number_of_contacts>0:
            PPO_mean_contact_energy_efficiency_vector[vector_index]=delivered_packets/(number_of_contacts*10)
        else:
            PPO_mean_contact_energy_efficiency_vector[vector_index]=0
        PPO_initial_data_volume[vector_index]=initial_data_volume*100
        if(np.round(initial_data_volume*100)<delivered_packets):
            print("Delivery ratio error PPO")
            print(initial_data_volume*100)
            print(delivered_packets)
        vector_index+=1
    envs.close()
    return PPO_reward_vector,PPO_delivery_ratio_vector,PPO_mean_contact_energy_efficiency_vector,PPO_initial_data_volume
def BaselineCGR(env,obs,init_info):
    delivery_ratio_metric=None#Delivery Ratio Metric
    contact_energy_efficiency=None#Contact Energy Efficiency metric
    total_reward=0
    done=False
    while not done:
        action = 1#We always choose action 1 until the episode is over
        obs, reward, terminated, truncated, info = env.step(action)
        print("Base action")
        done = terminated or truncated
        total_reward += reward
    #Next we need to extract final info from the episode
    initial_data_volume=info["Initial Data"]
    delivered_packets=info["Delivered Data"]
    number_of_contacts=info["Number of Contacts"]
    excess_energy_expended=info["Energy Expended"]
    if(np.round(initial_data_volume*100)<delivered_packets):
        print("Delivery ratio error Base")
        print(initial_data_volume*100)
        print(delivered_packets)
    return total_reward,initial_data_volume,delivered_packets,number_of_contacts,excess_energy_expended#Return key metrics
def AdaptiveSorting(env,init_obs,init_info):
    initial_observation,initial_information=init_obs,init_info
    #print(initial_observation)
    initial_data_volume=initial_observation[10]*100
    contacts_available=initial_observation[0:10]
    sorted_matrix=np.zeros([10,2])
    current_contact_iterator=0
    total_reward=0
  
    
    #Next we must calculate the initial allocation of contacts
    j=0
    for i in contacts_available:
        sorted_matrix[j,0]=i
        sorted_matrix[j,1]=j
        j=j+1
    sorted_matrix=sorted_matrix[sorted_matrix[:, 0].argsort()]
    #print("Initial Sorted Matrix")
    #print(sorted_matrix)
    remaining_data=initial_data_volume
    #print("Initial Data to send")
    #print(remaining_data)
    decision_matrix=np.zeros([10,2])
    
    j=0
    for i in range(0,len(decision_matrix)):
        if (remaining_data>0):
            if sorted_matrix[i,0]<1:
                remaining_data=remaining_data-(10-sorted_matrix[i,0]*10)
                
                decision_matrix[i,0]=1
                decision_matrix[i,1]=sorted_matrix[i,1]
            else:
                decision_matrix[i,0]=0
                decision_matrix[i,1]=sorted_matrix[i,1]
        else:
            decision_matrix[i,0]=0
            decision_matrix[i,1]=sorted_matrix[i,1]
    #Next we must reorder the decision matrix
    

    #print("Decision Matrix")
    #print(decision_matrix)
    sorted_decisions=decision_matrix[decision_matrix[:,1].argsort()]
    #print("Sorted Decisions")
    #print(sorted_decisions)
    action_decision_matrix=np.zeros(10)
    for i in range(0,len(sorted_decisions)):
        action_decision_matrix[i]=sorted_decisions[i,0]
    #print("Actual set of Actions")
    #print(action_decision_matrix)
    obs=initial_observation
    i=0
    done=False
    while not done:
        if action_decision_matrix[i]==0:
            #We don't use this contact
            action=0
            next_obs, reward, terminated, truncated, info = env.step(action)
            print("Sort action")
            #print("Action")
            #print(action)
            total_reward+=reward
        else:
            #We use the contact
            
            action=1
            #print(action)
            next_obs, reward, terminated, truncated, info = env.step(action)
            print("Sort action")
            total_reward+=reward
            #Next we need to determine if we successfully delivered any packets
            current_delivered=1-next_obs[10]
            #print("Current_Delivered")
            #print(current_delivered)
            previous_delivered=1-obs[10]
            #print(previous_delivered)
            if (current_delivered<previous_delivered):
                #We need to recalculate the decision matrix
                remaining_data=current_delivered*100
                aux_sorting_matrix=np.zeros([10,2])
                for j in range(0,10):
                    aux_sorting_matrix[j,0]=contacts_available[j]
                    aux_sorting_matrix[j,1]=j
                for j in range(0,i+1):
                    aux_sorting_matrix[j,0]=1
                aux_sorting_matrix=aux_sorting_matrix[aux_sorting_matrix[:, 0].argsort()]
                decision_matrix=np.zeros([10,2])
                k=0
                for j in range(0,len(decision_matrix)):
                    if remaining_data>0:
                        if aux_sorting_matrix[j,0]<1:
                            remaining_data=remaining_data-(10-aux_sorting_matrix[j,0]*10)
                            decision_matrix[j,0]=1
                            decision_matrix[j,1]=aux_sorting_matrix[j,1]
                        else:
                            decision_matrix[j,0]=0
                            decision_matrix[j,1]=aux_sorting_matrix[j,1]
                    else:
                        decision_matrix[j,0]=0
                        decision_matrix[j,1]=aux_sorting_matrix[j,1]

                aux_sorting_decisions=decision_matrix[decision_matrix[:,1].argsort()]
                action_decision_matrix=np.zeros(10)
                for k in range(0,len(aux_sorting_decisions)):
                    action_decision_matrix[k]=aux_sorting_decisions[k,0]
        i+=1
        obs=next_obs
        done = terminated or truncated
        #print(done)
    #Next we need to graph and log the results of the heuristic algorithm
    initial_data_volume=info["Initial Data"]
    #print("Sorting Algo")
    #print("Initial Data Volume")
    #print(initial_data_volume)
    delivered_packets=info["Delivered Data"]
    #print(delivered_packets)
    number_of_contacts=info["Number of Contacts"]
    excess_energy_expended=info["Energy Expended"]
    if((np.round(initial_data_volume*100))<delivered_packets):
        print("Delivery ratio error Sort")
        print(initial_data_volume*100)
        print(delivered_packets)
    return total_reward,initial_data_volume,delivered_packets,number_of_contacts,excess_energy_expended
    env.close()
#Static sorting
#Make all decisons prior to performing any actions
def StaticSorting(env,init_obs,init_info):
    





#Single Threshold

def SingleThreshold(env,init_obs,init_info,threshold):
    delivery_ratio_metric=None#Delivery Ratio Metric
    contact_energy_efficiency=None#Contact Energy Efficiency metric
    total_reward=0
    done=False
    weather_conditions=init_obs[]
    while not done:
        if weather_conditions>threshold:
            action=0
        else:
            action = 1
        obs, reward, terminated, truncated, info = env.step(action)
        print("Base action")
        done = terminated or truncated
        total_reward += reward
        weather_conditions=obs[]
    #Next we need to extract final info from the episode
    initial_data_volume=info["Initial Data"]
    delivered_packets=info["Delivered Data"]
    number_of_contacts=info["Number of Contacts"]
    excess_energy_expended=info["Energy Expended"]
    if(np.round(initial_data_volume*100)<delivered_packets):
        print("Delivery ratio error Base")
        print(initial_data_volume*100)
        print(delivered_packets)
    return total_reward,initial_data_volume,delivered_packets,number_of_contacts,excess_energy_expended#Return key metrics



#Multi-Threshold
def MultiThreshold(env,init_obs,init_info,thresholds):
    delivery_ratio_metric=None#Delivery Ratio Metric
    contact_energy_efficiency=None#Contact Energy Efficiency metric
    total_reward=0
    done=False
    initial_data_volume=init_obs[]
    
    while not done:
        if initial_data_volume<0.2:
            if weather_conditions>thresholds[0]:
                action=0
            else:
                action=1
        elif initial_data_volume>=0.2 and initial_data_volume<0.4:
            if weather_conditions>thresholds[1]:
                action=0
            else:
                action=1
        elif initial_data_volume>=0.4 and initial_data_volume<0.6:
            if weather_conditions>thresholds[2]:
                action=0
            else:
                action=1
        elif initial_data_volume>=0.6 and initial_data_volume<0.8:
            if weather_conditions>thresholds[3]:
                action=0
            else:
                action=1
        elif initial_data_volume>=0.8:
            if weather_conditions>thresholds[4]:
                action=0
            else:
                action=1
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
    #Next we need to extract final info from the episode
    initial_data_volume=info["Initial Data"]
    delivered_packets=info["Delivered Data"]
    number_of_contacts=info["Number of Contacts"]
    excess_energy_expended=info["Energy Expended"]
    if(np.round(initial_data_volume*100)<delivered_packets):
        print("Delivery ratio error Base")
        print(initial_data_volume*100)
        print(delivered_packets)
    return total_reward,initial_data_volume,delivered_packets,number_of_contacts,excess_energy_expended#Return key metrics
    

IndentationError: expected an indented block after function definition on line 291 (1734117681.py, line 299)